[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/cours/seance1_cours.ipynb)

# Séance 4.1 — Prédire un nombre — expliquer n'est pas prédire

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qui sépare un modèle qui explique d'un modèle qui prédit
- découper un jeu de données en apprentissage et test, et dire pourquoi
- mesurer une erreur de prédiction en euros avec la RMSE et la MAE
- reconnaître un surapprentissage à l'écart entre les deux jeux
- repérer une fuite de données — l'erreur qui donne un modèle parfait et inutile

## La même équation, une autre question

En séance 3.4, vous avez ajusté `ca ~ qte` et lu le coefficient : **1,44 € par
article**. C'était de l'**explication** — comprendre ce qui se passe dans les
données que vous avez.

Aujourd'hui, la question change :

> *« Une commande arrive : 150 unités, 12 produits distincts. Combien va-t-elle
> rapporter ? »*

C'est de la **prédiction** — se prononcer sur une ligne qu'on n'a jamais vue.
La même équation peut servir aux deux. Ce qui change, et c'est tout le sujet
de la séance, c'est **la façon de la juger**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

X = cmd[["qte", "nart"]]     ## ce qu'on connait avant de facturer
y = cmd["ca"]                ## ce qu'on veut prevoir

print(X.shape, y.shape)

Par convention, `X` désigne les variables d'entrée et `y` la cible. Toute la
suite du bloc utilise ces deux noms.

## 1. Un modèle qui se note lui-même

Commençons comme en séance 3.4 : on ajuste sur **tout** le fichier.

In [ ]:
modele = LinearRegression().fit(X, y)   ## ajuste sur TOUT le fichier

print("R2 :", round(r2_score(y, modele.predict(X)), 3))   ## note sur les memes

0,72. De quoi être satisfait — sauf que cette note a été calculée **sur les
lignes qui ont servi à construire le modèle**.

C'est réviser sur l'annale, puis passer cette annale comme examen. Le résultat
ne dit rien de ce qui se passera sur une commande nouvelle.

> **La règle du bloc :** un modèle se juge sur des données qu'il n'a jamais
> vues. Sinon, on ne mesure pas sa capacité à prédire, on mesure sa capacité à
> retenir.

## 2. Découper : apprentissage et test

On met de côté un quart des lignes. Le modèle ne les verra jamais pendant son
apprentissage, et c'est sur elles qu'on le notera.

In [ ]:
# random_state=67 : le tirage est aleatoire mais REPRODUCTIBLE. Le nombre
# lui-meme n'a aucune importance, seul compte qu'il soit FIXE : sans lui,
# votre voisin obtient d'autres chiffres que vous.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=67)

print(len(X_tr), "pour apprendre |", len(X_te), "pour noter")   ## 1466 et 489

In [ ]:
modele = LinearRegression().fit(X_tr, y_tr)   ## on apprend sur _tr

# La meme mesure des deux cotes : c'est l'ECART entre les deux qui parle
for nom, (Xs, ys) in {"apprentissage": (X_tr, y_tr), "test": (X_te, y_te)}.items():
    p = modele.predict(Xs)
    print(f"{nom:<14} MAE {mean_absolute_error(ys, p):7.1f}  "
          f"RMSE {mean_squared_error(ys, p) ** 0.5:7.1f}  R2 {r2_score(ys, p):.3f}")

### Lire ces trois nombres

- **MAE — 218 € sur le test.** L'erreur moyenne, en euros. « Nous nous
  trompons de 218 € sur une commande typique » se dit en réunion.
- **RMSE — 583 €.** La même idée, mais elle **punit les grosses erreurs** :
  se tromper une fois de 1 000 € coûte plus cher que dix fois de 100 €. Quand
  RMSE est très supérieure à MAE, c'est que quelques prédictions sont
  franchement ratées.
- **R² — 0,691.** Sans unité, donc incomparable d'un métier à l'autre.

### Le test est un peu moins bon, et c'est normal

Regardez les deux lignes : **183 € d'erreur en apprentissage, 218 € en test**.
**R² de 0,73 puis 0,69.** Le modèle est moins bon sur les lignes qu'il n'a
jamais vues.

C'est attendu, et ce n'est pas un défaut : les coefficients ont été **calculés
pour coller aux commandes d'apprentissage**, pas aux autres. Un modèle est
toujours un peu avantagé chez lui.

Ce qu'on surveille, ce n'est donc pas l'existence de l'écart, c'est sa
**taille**. Quatre points de R² : le modèle tient la route sur du nouveau.
Vous allez voir tout de suite à quoi ressemble un écart qui, lui, n'est pas
acceptable.

> 💡 **Demandez toujours les deux chiffres.** Un score annoncé sans préciser
> sur quel jeu il a été calculé n'est pas un score.

## 3. L'arbre de décision, et le surapprentissage

La régression trace **une droite**, la même pour tout le fichier. Un **arbre de
décision** procède autrement : il pose une suite de questions à seuil — *« la
commande dépasse-t-elle 265 unités ? »* — qui découpent les commandes en
groupes, et prédit pour chaque groupe **la moyenne de ce groupe**.

Sa prédiction n'est donc pas une droite : c'est un **escalier**.

In [ ]:
petites = cmd.query("qte < 600")   ## on zoome la ou sont 95 % des commandes
grille = pd.DataFrame({"qte": np.arange(0, 600, 5)})

plt.figure(figsize=(7, 4))
plt.scatter(petites["qte"], petites["ca"], s=6, alpha=0.25, label="commandes")

for prof in [1, 2, 5]:   ## une seule variable en entree, pour voir les marches
    arb = DecisionTreeRegressor(max_depth=prof, random_state=42)
    arb.fit(petites[["qte"]], petites["ca"])
    plt.plot(grille["qte"], arb.predict(grille), label=f"max_depth={prof}")

plt.axis([0, 600, 0, 2000])
plt.xlabel("qte : unites commandees")
plt.ylabel("ca : montant de la commande (euros)")
plt.legend()
plt.show()

### Ce que `max_depth` règle

`max_depth` est le **nombre de questions posées à la suite** avant de répondre.

| `max_depth` | Questions | Marches au maximum |
|---|---|---|
| 1 | 1 | 2 |
| 2 | 2 | 4 |
| 5 | 5 | 32 |

Chaque question supplémentaire **double** le nombre de marches possibles :
l'escalier peut épouser le nuage de plus en plus près. Sur la figure,
`max_depth=5` ne décrit plus une tendance, il suit les creux et les bosses.

C'est le réglage qui décide de la **souplesse** du modèle, et c'est exactement
ce qui va poser problème.

> 💡 On retrouvera les arbres en séance 4.3, où ils serviront à **décider**
> (« ce client va-t-il partir ? ») plutôt qu'à chiffrer un montant. On y lira
> alors leurs questions une par une.

### Un arbre sans aucune limite

Enlevons le garde-fou : autant de questions qu'il en veut, sur les deux
variables.

In [ ]:
arbre = DecisionTreeRegressor(random_state=42).fit(X_tr, y_tr)   ## sans limite

for nom, (Xs, ys) in {"apprentissage": (X_tr, y_tr), "test": (X_te, y_te)}.items():
    print(f"{nom:<14} R2 {r2_score(ys, arbre.predict(Xs)):.3f}")

**0,980 en apprentissage, 0,495 en test.**

L'arbre a appris les commandes **par cœur** au lieu d'apprendre la règle qui
les gouverne. Sur une commande nouvelle, sa mémoire ne lui sert à rien.

C'est le **surapprentissage**, et il se reconnaît toujours de la même façon :
un écart important entre les deux jeux. L'écart de quatre points de la
régression était normal ; celui-ci, **quarante-huit points**, ne l'est pas.

Maintenant, bridons-le.

In [ ]:
# max_depth=3 : trois questions au maximum avant de repondre
bride = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_tr, y_tr)

print("sans limite :", round(r2_score(y_te, arbre.predict(X_te)), 3))
print("profondeur 3:", round(r2_score(y_te, bride.predict(X_te)), 3))

**0,495 contre 0,597.** Le modèle le plus bridé prédit le mieux : trois
questions valent mieux qu'un nombre illimité.

> ⚠️ **Plus complexe ne veut pas dire meilleur.** C'est le contraire de
> l'intuition, et c'est la raison d'être de la moitié des techniques de ce
> bloc.

## 4. Deux erreurs, dont une qui ne prévient pas

### L'erreur bruyante

In [ ]:
r2_score(y_te, modele.predict(X_tr))   ## _te contre _tr : melange

Dernière ligne :

```
ValueError: Found input variables with inconsistent numbers of samples: [489, 1466]
```

489 vraies valeurs contre 1 466 prédictions : on compare le test aux
prédictions faites sur l'apprentissage. Les deux jeux ont été mélangés.

### L'erreur silencieuse

Ajoutons `ca` aux variables explicatives — c'est-à-dire la réponse elle-même.

In [ ]:
fuite = cmd[["qte", "nart", "ca"]]   ## "ca" est la reponse elle-meme
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(fuite, y, test_size=0.25, random_state=67)

parfait = LinearRegression().fit(Xf_tr, yf_tr)
print("R2 :", round(r2_score(yf_te, parfait.predict(Xf_te)), 6))

**R² = 1,0.** Prédiction parfaite. Et parfaitement inutile : on a donné la
réponse au modèle.

Ça s'appelle une **fuite de données**, et sous cette forme grossière personne
ne s'y laisse prendre. En entreprise elle est bien plus discrète : une
variable renseignée **après** l'événement qu'on prétend prévoir — un motif de
résiliation dans un modèle qui prédit la résiliation, une date de livraison
dans un modèle qui prédit le retard.

> ⚠️ **Devant un score trop beau, cherchez la fuite avant de sabler le
> champagne.** Pour chaque variable : *serait-elle connue au moment où je dois
> décider ?* Si non, elle n'a rien à faire dans le modèle.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| découper les données | `train_test_split(X, y, test_size=0.25, random_state=67)` |
| ajuster un modèle | `m = LinearRegression().fit(X_tr, y_tr)` |
| prédire | `m.predict(X_te)` |
| l'erreur moyenne, en euros | `mean_absolute_error(y_te, p)` |
| l'erreur qui punit les grosses fautes | `mean_squared_error(y_te, p) ** 0.5` |
| la part expliquée | `r2_score(y_te, p)` |
| un modèle plus souple | `DecisionTreeRegressor(max_depth=3)` |
| la souplesse de l'arbre | `max_depth` : le nombre de questions posées à la suite |

## Les quatre phrases à retenir

1. **Un modèle noté sur les données qui l'ont produit se note lui-même.**
   La seule note qui compte est celle obtenue sur des lignes qu'il n'a jamais vues.

2. **RMSE et MAE se lisent en euros.** « Je me trompe de 218 € en moyenne » est
   une phrase de gestion ; « R² = 0,69 » n'en est pas une.

3. **Surapprentissage = excellent en apprentissage, mauvais en test.** L'arbre
   sans limite passe de R² 0,98 à 0,50 d'un jeu à l'autre.

4. **Une variable qui contient la réponse donne un modèle parfait et inutile.**
   Avant de se réjouir d'un R² de 1, chercher la fuite.